# **CORRELAÇÃO E REGRESSÃO LINEAR SIMPLES**

# **Análise e Tratamento dos Dados**

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
mola = pd.read_csv('rigidez.csv',
                    sep=';', encoding='iso-8859-1')
# encoding: codificação de caracteres, normalmente utiliza-se o iso-8859-1, utf-8, latin-1)

**Objetivo: Analisar a correlação entre a força e a deformação sofrida por uma mola e criar um modelo de regressão linear.**

Visualizando as primeiras linhas

In [ ]:
mola.head()

Dimensão dos dados

In [ ]:
mola.shape

Dicionário de Variáveis:
- Cargas:
- Força: Força que a carga exerce na mola
- Lo: Cumpriemnto inicial
- L: Cumprimento final
- x: Deformação após a força
- K: Constante elástica ou rigidez = força / deformação

Para facilitar a compreensão vamos renomear para ficar claro

In [ ]:
mola.rename(columns={'Lo': 'compr_inicial','L': 'compr_final','x': 'deformacao', 'K': 'rigidez'}, inplace=True)

In [ ]:
mola.head()

Análise do tipo de variáveis

In [ ]:
mola.dtypes

In [ ]:
# Excluir variável
dados_1 = mola.drop(columns=['compr_inicial'])
dados_1.head()

Valores Missing (NAN) - Valores faltantes

In [ ]:
dados_1.isnull().sum()

Temos que realizar tratamento para valores faltantes.



*   Excluir a observação que contém valores faltantes
*   Substituir os valores faltantes por alguma medida de tendencia central



In [ ]:
dados_1 = dados_1.dropna()

In [ ]:
dados_1.head()

In [ ]:
dados_1.describe()

Criando novamente a base de dados com valores ausentes novamente

In [ ]:
dados_2 = mola.drop(columns=['compr_inicial'])

Substituindo pela mediana

In [ ]:
dados_2['compr_final'].fillna(dados_2['compr_final'].median(), inplace=True)
dados_2['deformacao'].fillna(dados_2['deformacao'].median(), inplace=True)
dados_2['rigidez'].fillna(dados_2['rigidez'].median(), inplace=True)

In [ ]:
dados_2.head()

Criando novamente a base de dados com valores ausentes novamente

In [ ]:
dados_3 = mola.drop(columns=['compr_inicial'])

Substituindo pela média

In [ ]:
dados_3['compr_final'].fillna(dados_3['compr_final'].mean(), inplace=True)
dados_3['deformacao'].fillna(dados_3['deformacao'].mean(), inplace=True)
dados_3['rigidez'].fillna(dados_3['rigidez'].mean(), inplace=True)

Análise dos outliers

In [ ]:
import plotly.express as px

In [ ]:
boxplot = px.box(dados_1, y="forca")
boxplot.show()

In [ ]:
boxplot = px.box(dados_1, y="deformacao")
boxplot.show()

In [ ]:
dados_1.head(30)

In [ ]:
boxplot = px.box(dados_1, y="rigidez")
boxplot.show()

In [ ]:
dados_1.head(30)

In [ ]:
dados_1.drop(28, inplace=True)

In [ ]:
dados_1.head(30)

# **ANÁLISE DA CORRELAÇÃO LINEAR**

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.scatter(dados_1.deformacao,dados_1.forca)
plt.title('Relação entre as variáveis')
plt.xlabel('Deformação')
plt.ylabel('Força')
plt.grid(False)
plt.show()

## Análise da Normalidade

Gráfico QQ-Plot

In [ ]:
import scipy.stats as stats

In [ ]:
stats.probplot(dados_1['forca'], dist="norm", plot=plt)
plt.title("Normal Q-Q plot")
plt.show()

In [ ]:
stats.probplot(dados_1['deformacao'], dist="norm", plot=plt)
plt.title("Normal Q-Q plot")
plt.show()

Teste Shapiro-Wilk

Ho = distribuição normal : p > 0.05

Ha = distribuição não é normal : p <= 0.05

In [ ]:
stats.shapiro(dados_1.forca)

In [ ]:
estatistica, p = stats.shapiro(dados_1.forca)
print('Estatística do teste: {}'.format(estatistica))
print('p-valor: {}'.format(p))

In [ ]:
estatistica, p = stats.shapiro(dados_1.deformacao)
print('Estatística do teste: {}'.format(estatistica))
print('p-valor: {}'.format(p))

Teste Lilliefors (Kolmogorov_Sminorv)

Ho = distribuição normal : p > 0.05

Ha = distribuição != normal : p <= 0.05

In [ ]:
import statsmodels
from statsmodels.stats.diagnostic import lilliefors

In [ ]:
estatistica, p = statsmodels.stats.diagnostic.lilliefors(dados_1.forca, dist = 'norm')
print('Estatística de teste: {}'.format(estatistica))
print('p-valor: {}'.format(p))

In [ ]:
estatistica, p = statsmodels.stats.diagnostic.lilliefors(dados_1.deformacao, dist = 'norm')
print('Estatística de teste: {}'.format(estatistica))
print('p-valor: {}'.format(p))

## Correlação Linear

Pearson (distribuição normal)

Spearman (distribuição não normal)

Kendall (distribuição não normal com quantidade pequena de amostras)

Ho = não há corrrelação linear: p > 0,05

Ha = existe correlação linear: p <= 0,05

In [ ]:
# Pearson
coef,p = stats.pearsonr(dados_1.deformacao,dados_1.forca)
print('Coeficiente de correlação: {}'.format(coef))
print('p-valor: {}'.format(p))

In [ ]:
# Spearman
coef,p = stats.spearmanr(dados_1.deformacao,dados_1.forca)
print('Coeficiente de correlação: {}'.format(coef))
print('p-valor: {}'.format(p))

In [ ]:
# Kendall
coef,p = stats.kendalltau(dados_1.deformacao,dados_1.forca)
print('Coeficiente de correlação: {}'.format(coef))
print('p-valor: {}'.format(p))

In [ ]:
correlacoes = dados_1.corr(method ='pearson')
correlacoes

In [ ]:
import seaborn as sns

In [ ]:
plt.figure()
sns.heatmap(correlacoes, annot= True)

# **MODELO DE REGRESSÃO LINEAR**

## Regressão Linear com Statsmodels

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms

In [ ]:
# Criação do modelo
regressao = smf.ols('forca ~ deformacao', data = dados_1).fit()

In [ ]:
residuos = regressao.resid
residuos

### 1º Pressuposto - Teste de Normalidade dos resíduos

Ho = distribuição normal : p > 0.05

Ha = distribuição != normal : p <= 0.05

In [ ]:
estatistica, p = stats.shapiro(residuos)
print('Estatística de teste: {}'.format(estatistica))
print('p-valor: {}'.format(p))

In [ ]:
stats.probplot(residuos, dist="norm", plot=plt)
plt.title("Normal Q-Q plot - Resíduos")
plt.show()

### 2º Pressuposto : Análise da Homocedasticidade dos resíduos

Os dados são homocedásticos quando tem variação constante

In [ ]:
plt.scatter(y=residuos, x=regressao.predict(), color='red')
plt.hlines(y=0, xmin=0, xmax=4, color='orange')
plt.ylabel('Resíduos')
plt.xlabel('Valores Preditos')
plt.show()

Teste Breusch-Pagan (Homocedasticidade ou heterocedasticidade)

Ho = existe homocedasticidade : p > 0.05

Ha = não existe homocedasticidade : p <= 0.05

In [ ]:
from statsmodels.compat import lzip
import statsmodels.stats.api as sms

In [ ]:
estatistica, p, f, fp = sms.het_breuschpagan (regressao.resid, regressao.model.exog)
print('Estatística de teste: {}'.format(estatistica))
print('p-valor: {}'.format(p))
print('f-valor: {}'.format(f))
print('f_p-valor: {}'.format(fp))

### **Outliers nos resíduos**

No teste de outliers os valores devem estar dentro do intervalo 3 e -3

In [ ]:
outliers = regressao.outlier_test()

In [ ]:
outliers.max()

In [ ]:
outliers.min()

**Modelo Verificado**

### **Regressão Linear**

In [ ]:
regressao.summary()


Estatística t:

Ho = coeficiente igual a zero : p > 0,05 (coeficiente não validado)

Ha = coeficiente diferente de zero: p <= 0,05 (coeficiente validado)

**Equação: Força = 0,0436 + 30,2326.deformação**

**R^2 ajustado = 0,998**

In [ ]:
coefs = pd.DataFrame(regressao.params)
coefs.columns = ['Coeficientes']
print(coefs)

In [ ]:
regressao.params

In [ ]:
dados_1.head()

In [ ]:
regressao.predict()

In [ ]:
plt.scatter(y=dados_1.forca, x=dados_1.deformacao, color='blue', s=80, alpha=0.9)
X_plot = np.linspace(0, 0.12)
plt.plot(X_plot, X_plot*regressao.params[1] + regressao.params[0], color='r')
plt.title('Reta de regressão')
plt.ylabel('FORÇA')
plt.xlabel('DEFORMAÇÃO')
plt.show()

## Regressão Linear com Sklearn

In [ ]:
dados_1.head()

In [ ]:
x = dados_1.iloc[ : , 3].values
y = dados_1.iloc[ : , 1].values

In [ ]:
correlacao2 = np.corrcoef (x, y)
correlacao2

In [ ]:
x

In [ ]:
x = x.reshape(-1,1) #transformando em matriz

In [ ]:
x

In [ ]:
from sklearn.linear_model import LinearRegression
regressao2 = LinearRegression()
regressao2.fit(x,y)

In [ ]:
regressao2.intercept_

In [ ]:
regressao2.coef_

In [ ]:
# coeficiente de determinação
regressao2.score(x,y)

**Equação: Força = 0,0436 + 30,2326.deformação**

**R^2 ajustado = 0,998**

In [ ]:
dados_1.head(20)

In [ ]:
previsoes = regressao2.predict(x)
previsoes

In [ ]:
previsao = regressao2.predict([[0.45]])
print('A força deve ser de {} N'.format(previsao))